In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
  
    mesaages = []

    add_user_message(mesaages, prompt)
    add_assistant_message(mesaages, '```json')

    text = chat(mesaages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()

# dataset

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then retruns the result"""
    prompt = f"""
Please slove the folowing task:

{test_case["task"]}

* Respond only wiht Python, JSON, or a plain Regex
* Do not add any comments or commentary or explaintion
"""
    messages =[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")  # ```code as a stop seq we dont have to specify py, json, regex.... etc.
    output = chat(messages, stop_sequences=["```"])
    return output

In [6]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [15]:
#  Functions to validate the output structure
import re
import ast   # ast --> Abstract Syntax TRee

def validate_json(text):
    try: 
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except re.error:
        return 0
    
def validate_regex(text):
    try: 
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)


    # Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    # Syntax grader
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "resoning": reasoning
    }

In [17]:
from statistics import mean

def run_eval(dataset):
    """Loads  the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        res = run_test_case(test_case)
        results.append(res)
    average_score = mean([result["score"] for result in results])
    print(f"Average_score: {average_score}")

    return results

In [18]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average_score: 8.0


In [ ]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Validator\n\nHere's a Python function that validates AWS S3 bucket names:\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name: str) -> bool:\n    \"\"\"\n    Validates if a bucket name follows AWS S3 naming conventions.\n    \n    AWS S3 Bucket Naming Rules:\n    - Must be between 3 and 63 characters long\n    - Can consist only of lowercase letters, numbers, and hyphens (-)\n    - Must start with a lowercase letter or number\n    - Must end with a lowercase letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted as an IP address (e.g., 192.168.1.1)\n    \n    Args:\n        bucket_name (str): The name of the S3 bucket to validate\n        \n    Returns:\n        bool: True if the bucket name is valid, False otherwise\n    \"\"\"\n    \n    # Check if bucket_name is a string\n    if not isinstance(bucket_name, str):\n        return False\n    \n    # Check length (3-63 characters)\n    if len(bucket_